---
title: Poisson Model of Timing Prediction Market Contracts
jupyter: python3
---

Consider a timing prediction market which resolves to *yes* if some given event occurs before a final time $T$. I propose modeling the event as arriving by a Poisson process with Ornstein-Uhlenbeck intensity $\lambda_t$. This gives rise to the following model:
$$
\begin{align}
\,\mathrm{d}\lambda_t &= -\kappa (\lambda _t-\mu )\,\mathrm{d}t +\sigma \,\mathrm{d}W_t \\
\pi _t &= \mathbb{E}\left[ e^{-\int_t^T \lambda_s \mathrm{d}s}\vert \mathcal{F}_t \right]  
\end{align}
$$
where $\pi_t$ is the *no*-contract price at time $t \leq T$. This model is parameterized by the volatity $\sigma>0$, mean reversion strength $\kappa>0$. The mean $\mu$ ought also to be positive to preserve interpretability as an intensity process. 

<div class="alert alert-block alert-warning">
<b>Clipping:</b> The OU-process is not inherently non-negative, which makes it a suboptimal model for an intensity process. I chose it anyway to facilitate derivations of the price dynamics. The process must then be clipped to $[0, \infty)$. With realistic parameter values, I do not expect this to introduce major inaccuracies.
</div>

If you work out the Itô calculus for this model, you get the following price dynamics:
$$
\,\mathrm{d}\pi_t = \pi_t\left[ \mu +L_1(\log\pi_t + L_2) \right]\,\mathrm{d}t + \pi_t \frac{\sigma (e^{-\kappa \tau }-1)}{\kappa }\,\mathrm{d}W_t,
$$
where 
$$
L_1 = \frac{\kappa }{e^{-\kappa \tau }-1}<0, \quad L_2=\mu \tau -\frac{V}{2}, \quad V = \frac{\sigma ^2}{2\kappa ^3}\left( 2\kappa \tau +4e^{-\kappa \tau }-e^{-2\kappa \tau } - 3 \right).
$$
Let's see how this model behaves! 


## Simulation
As always, we first simulate the model. Note that the above dynamics are conditional on the market not resolving. At each step we first need to check whether the event occurs before simulating the next innovation. Consider an Euler-Maruyama scheme with constant time step $\Delta$ and initial price $\pi_0$ for times $t_k = \Delta k$, $k=0, \dots, T/\Delta =: n$. We have
$$
\pi_{k+1} = \pi_k + \pi_k\Delta\left[\mu + L_1(\log\pi_k + L_2\right] + \pi_k \frac{\sigma}{\kappa}(e^{-\kappa\tau}-1)\mathcal{N(0, \Delta)},
$$
where $\tau:= T-t_k$. We here write $\pi_k \equiv \pi_{t_k}$, for the sake of convenience. The probability of the market resolving in the time interval $[t_k, t_{k+1}]$ is (after discretization)
$$
1-e^{-\lambda_t \Delta} = 1-\pi_t^{-\Delta L_1} e^{-\Delta(\mu + L_1L_2)}.
$$
We simulate this process:

In [ ]:
import numpy as np
from numpy import log, sqrt, exp
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from plot_style import (
    COLORS, FONT_SIZES, setup_matplotlib_style,
    create_styled_figure, style_legend, format_datetime_axis,
    save_styled_figure
)
rng = np.random.default_rng(42)

T0 = 0
T = 1
n = 1000

# model parameters
mu = 1
sigma = 1/2
kappa = 0.5
price_init = 0.5

def simulate_timing_market(mu, sigma, kappa, price_init, n, T0=0, T=1):
    Delta = (T - T0) / n

    price = np.zeros(n)
    price[0] = 1 - price_init # price_init should be a yes-price
    intensity = np.zeros(n)
    intensity[-1] = np.nan
    resolve_time = None
    for k in range(0, n-1):
        tk = Delta*k + T0
        tau = T - tk
        V = sigma**2 / (2 * kappa**3) * (2*kappa*tau + 4*exp(-kappa*tau) -exp(-2*kappa*tau) - 3)
        L1 = kappa / (exp(-kappa*tau) - 1)
        L2 = mu * tau - V / 2
    
        # check if markte resolves in [t_k, t_k+1]
        prob_res = 1 - price[k]**(-Delta * L1) * exp(-Delta * (mu + L1 * L2))
        if rng.random() < prob_res:
            resolve_time = tk
            break
    
        while True:
            drift = price[k] * Delta*(mu + L1*(log(price[k]) + L2))
            diffusion = price[k] * sigma / kappa * sqrt(Delta) * (exp(-kappa*tau) - 1) * rng.normal()
            proposed_price = price[k] + drift + diffusion
            # accept proposed_price if it does not imply a negative intenstiy
            # else throw it away and try again
            if proposed_price <= exp(-(mu / L1 + L2)):
                price[k+1] = price[k] + drift + diffusion
                intensity[k] = mu + L1 * (log(price[k]) + L2)
                break
            else:
                #print(f'Clipped at time {tk:.2f}. Retrying...')
                pass

    t = np.linspace(T0, T, num=n)
    return t, price, intensity, resolve_time, Delta

t, price, intensity, resolve_time, Delta = simulate_timing_market(mu, sigma, kappa, price_init, n)

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(8, 6),
    gridspec_kw={"height_ratios": [3, 2]}
)
fig.suptitle(
    f'''Yes-Timing Contract Simulation
$\\mu={mu:.2f}, \\sigma={sigma:.2f}, \\kappa={kappa:.2f}$
$t \\in [{T0:.2f}, {T:.2f}], n={n}$'''
)
# --- Top: price ---
ax1.plot(t, (1 - price) * 100)
ax1.set_title('Contract Price $\\pi_t$')
ax1.set_ylabel('Price  (US¢)')
ax1.set_ylim(-5, 105)

# --- Bottom: intensity ---
ax2.plot(t, intensity)
ax2.axhline(mu, color="grey", linestyle="dashed")
ax2.set_title('Price Implied Intensity $\\lambda_t$ (Mean in Grey)')
ax2.set_ylabel('Intensity')
ax2.set_xlabel('t')

plt.tight_layout()
plt.show()

if resolve_time:
    print(f"Market resolved to 'yes' at time {resolve_time:.2f}.")
else:
    print(f"Market resolved to 'no' at end time T={T}")

#plt.figure(figsize=(12, 5))
#plt.title(f'''Yes-Timing Contract Simulation ($\\mu={mu:.2f}, \\sigma={sigma:.2f}, \\kappa={kappa:.2f}$)''',fontsize=22)
#plt.plot(t, (1 - price) * 100, linewidth=4)
#plt.ylabel('Price  (US¢)')
#plt.ylim(-5, 105)
#plt.savefig('phd_presentation/simulation.pdf')
#plt.tight_layout()
#plt.show()

## Estimation
To avoid solving the price SDE for $\pi_t$, we will estimate the model parameters $(\mu, \sigma, \kappa)$ with the QMLE using the Euler-Maruyama approximate transition density
$$
\pi_{k+1}|\pi_k \sim \mathcal{N}\left( \pi_k + \pi_k\Delta\left[\mu + L_1(\log\pi_k + L_2\right], \frac{\pi_k^2\sigma^2\Delta }{\kappa ^2}(e^{-\kappa \tau}-1)^2 \right)
$$
This is non-linear enough that we need numerical optimization. As usual we minmize the negative log-likelihood. To enforce $\sigma, \kappa>0$ we optimize these on the log-domain. We have physical reasons to do this also for $\mu$, but I got numerically unstable results when doing that.

Initial values are tricky. $\kappa$ enters non-linearly everywhere, so we expect it to be the most difficult to estimate. Let's do a grid search for fixed values of $\kappa$, estimating only $\mu$ and $\sigma$. 

In [ ]:
from scipy.optimize import minimize

def neg_ll(theta, t, price, Delta, T, resolve_time):
    mu = theta[0]
    sigma = exp(theta[1])
    kappa = exp(theta[2])
    if resolve_time is None: resolve_time = T
    
    ll = 0
    for tk, price_k, price_kp1 in zip(t[:-1], price[:-1], price[1:]):
        if tk >= resolve_time: break
            
        tau = T - tk
        V = sigma**2 / (2 * kappa**3) * (2*kappa*tau + 4*exp(-kappa*tau) -exp(-2*kappa*tau) - 3)
        L1 = kappa / np.expm1(-kappa * tau)
        L2 = mu * tau - V / 2

        mean = price_k + price_k * Delta * (mu + L1*(log(price_k) + L2))
        std = price_k * sigma * sqrt(Delta) / kappa * abs(np.expm1(-kappa * tau))
        
        ll += -np.log(std) - (price_kp1 - mean)**2 / (2*std**2)
    
    return -ll

def neg_ll_fix_kappa(theta, kappa, t, price, Delta, T, resolve_time):
    mu = theta[0]
    sigma = exp(theta[1])
    if resolve_time is None: resolve_time = T
    
    ll = 0
    for tk, price_k, price_kp1 in zip(t[:-1], price[:-1], price[1:]):
        if tk >= resolve_time: break
            
        tau = T - tk
        V = sigma**2 / (2 * kappa**3) * (2*kappa*tau + 4*exp(-kappa*tau) -exp(-2*kappa*tau) - 3)
        L1 = kappa / np.expm1(-kappa * tau)
        L2 = mu * tau - V / 2

        mean = price_k + price_k * Delta * (mu + L1*(log(price_k) + L2))
        std = price_k * sigma * sqrt(Delta) / kappa * abs(np.expm1(-kappa * tau))
        
        ll += -np.log(std) - (price_kp1 - mean)**2 / (2*std**2)
    
    return -ll
    
def initializer(grid_size, kappa_min, kappa_max, t, price, Delta, T, resolve_time):
    kappa_grid = np.exp(np.linspace(np.log(kappa_min), np.log(kappa_max), grid_size)) # ChatGPT suggestion. Why?
    best_val = np.inf
    best_theta = None
    
    for kappa0 in kappa_grid:
        res = minimize(
            neg_ll_fix_kappa,
            [1, np.log(0.2)],
            args=(kappa0, t, price, Delta, T, resolve_time),
            method="BFGS"
        )
        if res.fun < best_val:
            best_val = res.fun
            best_theta = [res.x[0], np.exp(res.x[1]), kappa0]

    return best_theta, best_val
    
kappa_min, kappa_max = 1e-1, 5
grid_size = 10
theta0, best_val = initializer(grid_size, kappa_min, kappa_max, t, price, Delta, T, resolve_time)

print(f'Result of Grid Search for κ ∈ [{kappa_min:.2f}, {kappa_max:.2f}]')
print('-' * 50)
print(f'Min. neg. log-likelihood: {best_val:.0f}')
print(f'μ: {theta0[0]:.2f} (true: {mu:.2f})')
print(f'σ: {theta0[1]:.2f} (true: {sigma:.2f})')
print(f'κ: {theta0[2]:.2f} (true: {kappa:.2f})')

From these initial values, we can now do a full optimization:

In [ ]:
res = minimize(
    neg_ll,
    theta0,
    args=(t, price, Delta, T, resolve_time),
    method="BFGS",
    options={"disp": False, "gtol": 1e-6}
)
mle_neg_ll = res.fun
mu_hat, log_sigma_hat, log_kappa_hat = res.x
sigma_hat = np.exp(log_sigma_hat)
kappa_hat = np.exp(log_kappa_hat)

# 95 % Wald CIs per the Delta method
mu_std, log_sigma_std, log_kappa_std = np.sqrt(np.diag(res.hess_inv))
z = 1.96
mu_95CI = [mu_hat - z*mu_std, mu_hat + z*mu_std]
log_sigma_95CI = [log(sigma_hat) - z*log_sigma_std, log(sigma_hat) + z*log_sigma_std]
log_kappa_95CI = [log(kappa_hat) - z*log_kappa_std, log(kappa_hat) + z*log_kappa_std]
sigma_95CI = np.exp(log_sigma_95CI)
kappa_95CI = np.exp(log_kappa_95CI)

jac_mu, jac_log_sigma, jac_log_kappa = res.jac

print("Parameter estimation summary (QMLE)")
print(f"Gradient norm at optimum: {np.linalg.norm(res.jac):.2e}\n")

print(f"{'Parameter':<10} {'Estimate':>12} {'95% Wald CI':>12} {'Jacobian':>12}")
print("-" * 62)

print(
    f"{'mu':<10} {mu_hat:12.2f} "
    f"[{mu_95CI[0]:.2f}, {mu_95CI[1]:.2f}] "
    f"{jac_mu:12.2e}"
)
print(
    f"{'sigma':<10} {sigma_hat:12.2f} "
    f"[{sigma_95CI[0]:.2f}, {sigma_95CI[1]:.2f}] "
    f"{jac_log_sigma:12.2e}"
)
print(
    f"{'kappa':<10} {kappa_hat:12.2f} "
    f"[{kappa_95CI[0]:.2f}, {kappa_95CI[1]:.2f}] "
    f"{jac_log_kappa:12.2e}\n"
)

<div class="alert alert-block alert-warning">
<b>CI:</b> Given the low quality of the estimates, I wouldn't really trust Wald CI's. It's feasible to compute profile likelihood-based CI's, which I'd rather like. Future work!  
</div>

We can get a better idea of the estimation quality by simulating a large number of time series with the same parameters and checking the average estimation error.

In [ ]:
from rich.console import Console
from rich.spinner import Spinner
from rich.live import Live

num_sims = 1000
n = 1000
mu = 1
sigma = 1/2
kappa = 0.5
price_init = 0.5

mu_est, sigma_est, kappa_est = [], [], []
console = Console()
spinner = Spinner("dots", text="Running simulations...")
with Live(spinner, console=console, refresh_per_second=10):
    for i in range(num_sims):
        spinner.text = f"Simulation {i+1}/{num_sims}"

        while True:
            t, price, _, resolve_time, Delta = simulate_timing_market(
            mu, sigma, kappa, price_init, n
            )
            if resolve_time is None: resolve_time = T

            if resolve_time >= 0.5: break

        theta0, _ = initializer(10, 1e-1, 5, t, price, Delta, T, resolve_time)
        res = minimize(
            neg_ll,
            theta0,
            args=(t, price, Delta, T, resolve_time),
            method="BFGS",
            options={"disp": False, "gtol": 1e-6}
        )

        mu_hat, log_sigma_hat, log_kappa_hat = res.x
        sigma_hat = np.exp(log_sigma_hat)
        kappa_hat = np.exp(log_kappa_hat)

        mu_est.append(mu_hat)
        sigma_est.append(sigma_hat)
        kappa_est.append(kappa_hat)

console.print("[green]✔ Done.[/green]")

In [ ]:
# --- compute means ---
mu_mean = np.mean(mu_est)
kappa_mean = np.mean(kappa_est)
sigma_mean = np.mean(sigma_est)

# Create mask for outliers with abs value > 50
mask = (np.abs(np.array(mu_est)) < 50) & (np.abs(np.array(sigma_est)) < 50) & (np.abs(np.array(kappa_est)) < 50)

num_masked = len(mu_est) - sum(mask)
# Apply mask to filter out outliers
mu_est_filtered = np.array(mu_est)[mask]
sigma_est_filtered = np.array(sigma_est)[mask]
kappa_est_filtered = np.array(kappa_est)[mask]

mu_mean_filtered = np.mean(mu_est_filtered)
sigma_mean_filtered = np.mean(sigma_est_filtered)
kappa_mean_filtered = np.mean(kappa_est_filtered)

mu_std_filtered = np.std(mu_est_filtered)
sigma_std_filtered = np.std(sigma_est_filtered)
kappa_std_filtered = np.std(kappa_est_filtered)

print(f"Summary of {num_sims} Parameter Estimations ({num_masked} / {len(mu_est)} Outliers Removed)\n")
print(f"{'Parameter':<10} {'True':>12} {'Estimate':>12} {'Std':>12}")
print("-" * 62)

print(f"{'mu':<10} {mu:12.6f} {mu_mean_filtered:12.6f} {mu_std_filtered:12.6f}")
print(f"{'sigma':<10} {sigma:12.6f} {sigma_mean_filtered:12.6f} {sigma_std_filtered:12.6f}")
print(f"{'kappa':<10} {kappa:12.6f} {kappa_mean_filtered:12.6f} {kappa_std_filtered:12.6f}")

# --- plot histograms ---
plt.figure(figsize=(12, 3))

plt.subplot(1, 3, 1)
plt.hist(mu_est, bins=30, density=True)
plt.axvline(mu_mean, linestyle="--")
plt.title("mu")
plt.xlabel("estimate")

plt.subplot(1, 3, 2)
plt.hist(kappa_est, bins=30, density=True)
plt.axvline(kappa_mean, linestyle="--")
plt.title("kappa")
plt.xlabel("estimate")

plt.subplot(1, 3, 3)
plt.hist(sigma_est, bins=30, density=True)
plt.axvline(sigma_mean, linestyle="--")
plt.title("sigma")
plt.xlabel("estimate")

plt.tight_layout()
plt.show()

This isn't great... $\mu$ is fairly well-identified on average, and $\sigma$ is not terrible. But $\kappa$ is, as expected very difficult. Maybe you'd want to do something iterative, alternating between fitting $(\mu, \sigma)$ and $\kappa$. Future work!

## Diagnostics
Let's implement some goodness-of-fit checks to see how well this model performs. A good first step is to consider the normalized returns
$$
r_k := \pi_k - \pi_{k-1}.
$$
By the Euler-Maruyama scheme, they should be approximately distributed like
$$
r_k \sim \mathcal{N}\left(\pi_k\Delta\left[\mu + L_1(\log\pi_k + L_2\right], \pi_k^2 \frac{\sigma^2}{\kappa^2}(e^{-\kappa\tau}-1)^2\Delta \right).
$$
We can then check the standardized residuals
$$
\frac{r_k - \mathbb{E}[r_k|\pi_k]}{\sqrt{\mathrm{Var}(r_k|\pi_k)}} \sim \mathcal{N}(0,1).
$$

In [ ]:
returns_norm = np.zeros(n-1)
for k in range(0, n-1):
    tk = Delta*k + T0
    tau = T - tk
    V = sigma_hat**2 / (2 * kappa_hat**3) * (2*kappa_hat*tau + 4*exp(-kappa_hat*tau) - exp(-2*kappa_hat*tau) - 3)
    L1 = kappa_hat / (exp(-kappa_hat*tau) - 1)
    L2 = mu_hat * tau - V / 2

    rk = price[k+1] - price[k]
    mean = price[k] * Delta * (mu_hat + L1 * (log(price[k]) + L2))
    std = abs(price[k] * sigma_hat / kappa_hat * sqrt(Delta) * np.expm1(-kappa_hat * tau))
    rk_norm = (rk - mean) / std

    returns_norm[k] = rk_norm

from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# histogram
axes[0, 0].hist(returns_norm, bins=30, density=True, alpha=0.6, color='g')
axes[0, 0].set_title('Standardized residuals')
axes[0, 0].set_xlabel('Residual value')
axes[0, 0].set_ylabel('Density')

# QQ-plot
stats.probplot(returns_norm, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title("QQ-plot of Standardized Residuals")

# ACF of residuals
nlags = min(40, len(returns_norm) - 1)
plot_acf(returns_norm, lags=nlags, ax=axes[1, 0])
axes[1, 0].set_title('ACF of standardized residuals')

# ACF of squared residuals
plot_acf(returns_norm**2, lags=nlags, ax=axes[1, 1])
axes[1, 1].set_title('ACF of squared standardized residuals')

plt.tight_layout()
plt.show()

# concise summary table
mean_all = np.mean(returns_norm)
median_all = np.median(returns_norm)
std_all = np.std(returns_norm)
n_total = returns_norm.size
pct_gt2 = 100 * np.mean(np.abs(returns_norm) > 2)

print("Standardized residuals summary")
print(f"{'Metric':<25}{'Value':>12}")
print("-" * 37)
print(f"{'Mean':<25}{mean_all:12.4f}")
print(f"{'Median':<25}{median_all:12.4f}")
print(f"{'Std':<25}{std_all:12.4f}")
print(f"{'Count':<25}{n_total:12d}")
print(f"{'>2σ (%)':<25}{pct_gt2:12.1f}")

Gorgeous, just as expected from simulated data! Finally, let's check the calibration by checking the coverage of CI's for the one-step-ahead predictions.

In [ ]:
def CI_coverage(level):
    alpha = 1 - level
    covered_count = 0
    tested_count = 0
    for k in range(0, n-1):
        tk = Delta*k + T0
        tau = T - tk
        V = sigma_hat**2 / (2 * kappa_hat**3) * (2*kappa_hat*tau + 4*exp(-kappa_hat*tau) - exp(-2*kappa_hat*tau) - 3)
        L1 = kappa_hat / (exp(-kappa_hat*tau) - 1)
        L2 = mu_hat * tau - V / 2

        if tk >= resolve_time: break

        mean = price[k] + price[k] * Delta * (mu_hat + L1 * (log(price[k]) + L2))
        std = abs(price[k] * sigma_hat / kappa_hat * sqrt(Delta) * np.expm1(-kappa_hat * tau))
        z = stats.norm.ppf(1 - alpha/2)

        tested_count += 1
        if price[k+1] > mean - z*std and price[k+1] < mean + z*std: covered_count += 1
    return covered_count / tested_count
    
levels = np.linspace(0.01, 0.99, 50)
coverages = [CI_coverage(level) for level in levels]

plt.figure(figsize=(8, 5))
plt.plot(levels, coverages, label='Empirical coverage')
plt.plot(levels, levels, 'k--', label='Perfect calibration')
plt.xlabel('Nominal coverage level')
plt.ylabel('Empirical coverage')
plt.title('Confidence Interval Coverage')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Reconstruction and Forecasting
With fitted parameters, we can start reconstructing and forecasting intensities, volatilities and prices.

### Reconstructed Intensity
Since our model gives us an invertible relationship between the price $\pi_t$ and intensity $\lambda_t$
$$
\lambda_t = \mu + L_1(\log \pi_t + L_2), 
$$
where 
$$
L_1 = \frac{\kappa }{e^{-\kappa \tau }-1}<0, \quad L_2=\mu \tau -\frac{V}{2}, \quad V = \frac{\sigma ^2}{2\kappa ^3}\left( 2\kappa \tau +4e^{-\kappa \tau }-e^{-2\kappa \tau } - 3 \right).
$$
We can use our parameter estimates to reconstruct past intensities. In principle one could use the $\Delta$-method to compute Wald CI's also for these intensities. It could also be done with Monte Carlo simulation, sampling from the approximate Gaussian distribution of the estimators. For now I'll do neither.

In [ ]:
intensity_hat = np.zeros(n-1)
for k, (tk, price_k) in enumerate(zip(t[:-1], price[:-1])):
    tau = T - tk
    V = sigma_hat**2 / (2 * kappa_hat**3) * (2*kappa_hat*tau + 4*exp(-kappa_hat*tau) -exp(-2*kappa_hat*tau) - 3)
    L1 = kappa_hat / np.expm1(-kappa_hat*tau)
    L2 = mu_hat * tau - V / 2
    intensity_hat[k] = mu_hat + L1 * (log(price_k) + L2)

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(8, 6),
    gridspec_kw={"height_ratios": [2, 3]}
)
fig.suptitle(
    f'''Intensity Reconstruction for a Yes-Timing Contract
$\\mu={mu:.2f}, \\sigma={sigma:.2f}, \\kappa={kappa:.2f}$
$t \\in [{T0:.2f}, {T:.2f}], n={n}$'''
)
# --- Top: price ---
ax1.plot(t, (1 - price) * 100)
ax1.set_title('Contract Price $\\pi_t$')
ax1.set_ylabel('Price  (US¢)')
ax1.set_ylim(-5, 105)

# --- Bottom: intensity ---
ax2.plot(t, intensity, label='True')
ax2.plot(t[:-1], intensity_hat, label='Reconstructed')
ax2.set_title('Intensity')
ax2.set_ylabel('Intensity')
ax2.set_xlabel('t')
ax2.legend()

plt.tight_layout()
plt.show()

if resolve_time:
    print(f"Market resolved to 'yes' at time {resolve_time:.2f}.")
else:
    print(f"Market resolved to 'no' at end time T={T}")

The reconstructed intensity is often negative. Not sure how much that should bother me. Probably just means that this reconstruction aren't particularly interesting.

## Reconstructed Volatility
Our Poisson model implies an instantaneous price volatility 
$$
\pi_t \frac{\sigma (e^{-\kappa \tau }-1)}{\kappa },
$$
which we can reconstruct using our parameter estimates. Again, leave CIs for future work.

In [ ]:
vol_hat = np.zeros(n)
vol = np.zeros(n)
for k, (tk, price_k) in enumerate(zip(t, price)):
    tau = T - tk
    vol[k] = abs(price_k * sigma / kappa * np.expm1(-kappa*tau))
    vol_hat[k] = abs(price_k * sigma_hat / kappa_hat * np.expm1(-kappa_hat*tau))
vol_realized = np.sqrt(np.pi / 2) *  abs(np.diff(price)) / sqrt(Delta)

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(8, 6),
    gridspec_kw={"height_ratios": [2, 3]}
)
fig.suptitle(
    f'''Volatility Reconstruction for a Yes-Timing Contract 
$\\mu={mu:.2f}, \\sigma={sigma:.2f}, \\kappa={kappa:.2f}$
$t \\in [{T0:.2f}, {T:.2f}], n={n}$'''
)
ax1.plot(t, (1 - price) * 100)
ax1.set_title('Contract Price $\\pi_t$')
ax1.set_ylabel('Price  (US¢)')
ax1.set_ylim(-5, 105)

ax2.plot(t[:-1], vol_realized, color='green', label='Realized', alpha=0.3)
ax2.plot(t, vol, label='True')
ax2.plot(t, vol_hat, label='Reconstructed')
ax2.set_title('Volatility')
ax2.set_ylabel(r'Volatility $\left(\text{US¢} \times \sqrt{\mathrm{time}}\,\right)$')
ax2.set_xlabel('t')
ax2.legend()

plt.tight_layout()
plt.show()

if resolve_time:
    print(f"Market resolved to 'yes' at time {resolve_time:.2f}.")
else:
    print(f"Market resolved to 'no' at end time T={T}")

## Probability of Resolution Within a Give Horizon
Say that we are observing an active market at some time $t_0$. Given parameter estimates, I can run a large number of simulations starting at the current market price. This allows me to predict a number of things. We will first estimate the probability of the market resolving within a given horizion, e.g. the next week. This is novel information not immediately availiable throught the market price! Our estimate is simply the proportion of simulations which resolves within the horizon.

In [ ]:
i0 = 500
t0 = t[i0]
price0 = 1-price[i0]
num_mc_sims = 1000

res_times = np.zeros(num_mc_sims)
for k in range(num_mc_sims):
    _, _, _, res_t, _ = simulate_timing_market(mu_hat, sigma_hat, kappa_hat, price0, n-i0, T0=t0, T=T)
    if res_t is None:
        res_times[k] = np.inf
    else:
        res_times[k] = res_t
        
horizon_grid = np.linspace(0, T-t0, num=1000)
prob_grid = [100*np.mean(res_times <= t0 + horizon) for horizon in horizon_grid]

plt.plot(horizon_grid, prob_grid)
plt.axhline(price0*100, color="grey", linestyle='--')
plt.ylabel('Cumulative Probability of Resolution (%)')
plt.xlabel(f'Horizon (time)')
#plt.ylim(-5, 105)
plt.title(rf'''Model-Implied Probability of Market Resolution
Within Horizon From $t_0 = {t0:.2f}$
(Market Price at time $t_0$ in Grey)''', fontsize=22)
plt.show()

To avoid leaking data, we first reestimate parameters with the data avaliable up to $t_0$.

In [ ]:
kappa_min, kappa_max = 1e-1, 5
grid_size = 10
theta0, best_val = initializer(grid_size, kappa_min, kappa_max, t[:i0], price[:i0], Delta, T, resolve_time)

print(f'Result of Grid Search for κ ∈ [{kappa_min:.2f}, {kappa_max:.2f}]')
print('-' * 50)
print(f'Min. neg. log-likelihood: {best_val:.0f}')
print(f'μ: {theta0[0]:.2f}')
print(f'σ: {theta0[1]:.2f}')
print(f'κ: {theta0[2]:.2f}')

res = minimize(
    neg_ll,
    theta0,
    args=(t[:i0], price[:i0], Delta, T, resolve_time),
    method="BFGS",
    options={"disp": False, "gtol": 1e-6}
)
mle_neg_ll = res.fun
mu_hat, log_sigma_hat, log_kappa_hat = res.x
sigma_hat = np.exp(log_sigma_hat)
kappa_hat = np.exp(log_kappa_hat)

# 95 % Wald CIs per the Delta method
mu_std, log_sigma_std, log_kappa_std = np.sqrt(np.diag(res.hess_inv))
z = 1.96
mu_95CI = [mu_hat - z*mu_std, mu_hat + z*mu_std]
log_sigma_95CI = [log(sigma_hat) - z*log_sigma_std, log(sigma_hat) + z*log_sigma_std]
log_kappa_95CI = [log(kappa_hat) - z*log_kappa_std, log(kappa_hat) + z*log_kappa_std]
sigma_95CI = np.exp(log_sigma_95CI)
kappa_95CI = np.exp(log_kappa_95CI)

jac_mu, jac_log_sigma, jac_log_kappa = res.jac

print("Parameter estimation summary (QMLE)")
print(f"Gradient norm at optimum: {np.linalg.norm(res.jac):.2e}\n")

print(f"{'Parameter':<10} {'Estimate':>12} {'95% Wald CI':>12} {'Jacobian':>12}")
print("-" * 62)

print(
    f"{'mu':<10} {mu_hat:12.2f} "
    f"[{mu_95CI[0]:.2f}, {mu_95CI[1]:.2f}] "
    f"{jac_mu:12.2e}"
)
print(
    f"{'sigma':<10} {sigma_hat:12.2f} "
    f"[{sigma_95CI[0]:.2f}, {sigma_95CI[1]:.2f}] "
    f"{jac_log_sigma:12.2e}"
)
print(
    f"{'kappa':<10} {kappa_hat:12.2f} "
    f"[{kappa_95CI[0]:.2f}, {kappa_95CI[1]:.2f}] "
    f"{jac_log_kappa:12.2e}\n"
)

These estimates are pretty similar to the ones on the entire data! 

In [ ]:
num_mc_sims = 1000

res_times = np.zeros(num_mc_sims)
for k in range(num_mc_sims):
    _, _, _, res_t, _ = simulate_timing_market(mu_hat, sigma_hat, kappa_hat, price0, n-i0, T0=t0, T=T)
    if res_t is None:
        res_times[k] = np.inf
    else:
        res_times[k] = res_t
        
horizon_grid = np.linspace(0, T-t0, num=1000)
prob_grid = [100*np.mean(res_times <= t0 + horizon) for horizon in horizon_grid]

plt.figure(figsize=(9, 9))
plt.plot(horizon_grid/Delta/10, prob_grid, linewidth=4)
plt.axhline(price0*100, color="grey", linestyle='--', label='Market Price at $t_0$', linewidth=4)
plt.ylabel('Cumulative Probability of Resolution (%)', fontsize=18)
plt.xlabel(f'Horizon (days)', fontsize=18)
plt.title(rf'''Probability of Market Resolution 
Within Horizon From $t_0 = {t0:.2f}$
(Simulated Data)''', fontsize=22)
plt.ylim(-2, price0*100 + 2)
plt.legend(fontsize=18)
plt.tick_params(axis="both", which="major", labelsize=16)
path = Path.home() / 'Obsidian/Courses/MASM12 Nonlinear Timeseries/MASM12-project/prob_sim.pdf'
#plt.savefig(path)
plt.show()

The model is pretty well calibrated! Is there a way to measure the uncertainty of these probabilities? Maybe with a boostrap steup... 

The above plot is produced conditional on $\pi_{t_0}, R>t_0$, where $R$ is the market resolution time. We can answer questions like 'What is the probability that the market resolves during the time interval [0.6,0.7]?' by instead conditioning on $\pi_{t_0}, R>0.6$.

<div class="alert alert-block alert-warning">
The above implementation is leaking data, because it uses parameter estimates derived from the entire time series. Make sure to avoid this when looking at real data.
</div>

## Predicting Future Prices, Intensities, and Volatilities
The same Monte Carlo setup as above gives us predictions for future prices, and thus also for intensities and volatilities. They can be given nice CI's using the quantiles of the MC samples.

In [ ]:
i0 = 300 # starting index
t0 = t[i0]
price0 = 1-price[i0]
num_mc_sims = 1000
horizon = 0.3
horizon_index = i0 + int(horizon/Delta) + 1

t_MC = np.zeros([num_mc_sims, n-i0])
price_MC = np.zeros([num_mc_sims, n-i0])
intensity_MC = np.zeros([num_mc_sims, n-i0])
vol_MC = np.zeros([num_mc_sims, n-i0])
resolve_time_MC = np.zeros(num_mc_sims)
Delta_MC = np.zeros(num_mc_sims)
for i in range(num_mc_sims):
    while True:
        t_sample, price_sample, intensity_sample, resolve_time_sample, Delta_sample = simulate_timing_market(mu_hat, sigma_hat, kappa_hat, 1-price0, n-i0, T0=t0, T=T)
        if resolve_time_sample is None: resolve_time_sample = np.inf
        if resolve_time_sample >= t0+horizon: break
        
    for k, (tk, price_k) in enumerate(zip(t_sample, price_sample)):
        tau = T - tk
        vol_MC[i, k] = abs(price_k * sigma_hat / kappa_hat * np.expm1(-kappa_hat*tau))
    
    t_MC[i, :] = t_sample
    price_MC[i, :] = price_sample
    intensity_MC[i, :] = intensity_sample
    resolve_time_MC[i] = resolve_time_sample
    Delta_MC[i] = Delta_sample

price_mean = np.mean(price_MC, axis=0)
price_low, price_high = np.quantile(price_MC, [0.025, 0.975], axis=0)
intensity_mean = np.mean(intensity_MC, axis=0)
intensity_low, intensity_high = np.quantile(intensity_MC, [0.025, 0.975], axis=0)
vol_mean = np.mean(vol_MC, axis=0)
vol_low, vol_high = np.quantile(vol_MC, [0.025, 0.975], axis=0)


fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1,
    sharex=True,
    figsize=(8, 6),
    gridspec_kw={"height_ratios": [3, 2, 2]}
)
fig.suptitle(f'Conditional Monte Carlo Forecasts of Contract Price, Event Intensity, and Volatility\nfrom t={t0:.2f} with Forecast Horizon of {horizon:.2f} ({num_mc_sims} Simulations)')

ax1.axvline(t0, color='grey', linestyle='--')
ax1.plot(t[:horizon_index], (1 - price[:horizon_index]) * 100, label='True')
ax1.plot(t[i0:horizon_index], (1-price_mean[:horizon_index-i0])*100, color='orange', label='MC Prediction')
ax1.fill_between(t[i0:horizon_index], 100*(1-price_low[:horizon_index-i0]), 100*(1-price_high[:horizon_index-i0]), color='orange', alpha=0.5, edgecolor='none')
ax1.set_title('Contract Price With MC Predictions')
ax1.set_ylabel('Price  (US¢)')
ax1.set_ylim(-5, 105)
ax1.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False
)

ax2.axvline(t0, color='grey', linestyle='--')
ax2.plot(t[:horizon_index], intensity[:horizon_index], color='#1f77b4', linestyle='--', label='True')
ax2.plot(t[:horizon_index], intensity_hat[:horizon_index], color='#1f77b4', label='Reconstructed')
ax2.plot(
    t[i0:horizon_index],
    intensity_mean[:horizon_index-i0],
    color='orange',
    label='MC Prediction'
)
ax2.fill_between(
    t[i0:horizon_index],
    intensity_low[:horizon_index-i0],
    intensity_high[:horizon_index-i0],
    color='orange',
    alpha=0.5,
    edgecolor='none'
)
ax2.set_title('Intensity With MC Predictions')
ax2.set_ylabel('Intensity')
ax2.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False
)

ax3.axvline(t0, color='grey', linestyle='--')
ax3.plot(t[:horizon_index], vol_hat[:horizon_index], label='Reconstructed')
ax3.plot(t[:horizon_index], vol[:horizon_index], color='#1f77b4', linestyle='dashed', label='True')
ax3.plot(
    t[i0:horizon_index],
    vol_mean[:horizon_index-i0],
    color='orange',
    label='MC Prediction'
)
ax3.fill_between(
    t[i0:horizon_index],
    vol_low[:horizon_index-i0],
    vol_high[:horizon_index-i0],
    color='orange',
    alpha=0.5,
    edgecolor='none'
)
ax3.set_title('Volatility With MC Predictions')
ax3.set_ylabel('Volatility')
ax3.set_xlabel('t')
ax3.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False
)
plt.tight_layout()
plt.show()

## Exact Fine-Grained Prediction Probabilities
Instead of using Monte Carlo simulations to estimate resolution probabilities, we can compute them exactly using the analytical solution we derived. This approach should be both faster and more accurate than simulation-based methods.

The exact probability of resolution within a time horizon $h$ from time $t_0$ is given by:

$$P(\text{Resolve by } t_0+h | \pi_{t_0}) = 1 - \exp(-D(\pi_{t_0}, h))$$

where the cumulant function is:

$$D(\pi_{t_0}, h) = \mu h + (\lambda_{t_0} - \mu) \times \frac{1-e^{-\kappa h}}{\kappa}$$

The current intensity $\lambda_{t_0}$ is obtained from the current market price $\pi_{t_0}$ through the invertible relationship established in our model.

In [ ]:
def compute_exact_resolution_probability(price0, h, mu, kappa, sigma, T, t0=0):
    """
    Compute exact fine-grained prediction probability using analytical solution.
    
    Parameters:
    price0: Current market price at time t0 (no-price)
    h: Time horizon for prediction (T - t0)
    mu, kappa, sigma: Model parameters
    T: Final time
    t0: Current time
    
    Returns:
    Probability of resolution within the time horizon
    """
    # Compute current intensity from current price using invertible relationship
    tau = T - t0  # This is what we called tau in the original formulation
    
    V_tau = sigma**2 / (2 * kappa**3) * (2*kappa*tau + 4*exp(-kappa*tau) - exp(-2*kappa*tau) - 3)
    V_h = sigma**2 / (2 * kappa**3) * (2*kappa*h + 4*exp(-kappa*h) - exp(-2*kappa*h) - 3)
    L2 = mu * tau - V_tau / 2
    D = mu*h - (1 - exp(-kappa*h)) / (1 - exp(-kappa*tau)) * (log(price0) + L2)

    prob = 1 - np.exp(V_h/2-D)
    
    return prob

# Example usage with our simulated data
i0 = 500
t0 = t[i0]
price0 = 1 - price[i0]  # yes-price
no_price0 = price[i0]   # no-price

# Compare exact probability with Monte Carlo estimate
horizon = 0.2  # 20% of total time horizon

# Exact probability
exact_prob = compute_exact_resolution_probability(
    no_price0, horizon, mu_hat, kappa_hat, sigma_hat, T, t0
)

# Compare with Monte Carlo approach
num_mc_sims = 1000
res_times = np.zeros(num_mc_sims)
for k in range(num_mc_sims):
    _, _, _, res_t, _ = simulate_timing_market(mu_hat, sigma_hat, kappa_hat, price0, n-i0, T0=t0, T=T)
    if res_t is None:
        res_times[k] = np.inf
    else:
        res_times[k] = res_t
        
mc_prob = np.mean(res_times <= t0 + horizon)

print(f"Exact probability of resolution within horizon {horizon:.2f}: {exact_prob:.4f}")
print(f"Monte Carlo estimate: {mc_prob:.4f}")
print(f"Difference: {abs(exact_prob - mc_prob):.6f}")

# Plot comparison over a range of horizons
horizon_grid = np.linspace(0, T-t0, num=100)
exact_prob_grid = [100*
    compute_exact_resolution_probability(
        no_price0, horizon, mu_hat, kappa_hat, sigma_hat, T, t0
    ) for horizon in horizon_grid
]

# Monte Carlo grid for comparison
prob_grid_mc = [100*np.mean(res_times <= t0 + horizon) for horizon in horizon_grid]

plt.figure(figsize=(10, 6))
plt.plot(horizon_grid, exact_prob_grid, 'b-', linewidth=2, label='Exact Solution')
plt.plot(horizon_grid, prob_grid_mc, 'r--', linewidth=2, label='Monte Carlo')
plt.axhline(price0*100, color="grey", linestyle='--', label='Market Price at $t_0$', linewidth=1)
plt.xlabel('Time Horizon')
plt.ylabel('Probability of Resolution')
plt.title(f'Comparison of Exact vs Monte Carlo Resolution Probabilities\n(Current time t₀ = {t0:.2f}, Current price = {price0:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Fitting Model to Real Data
We are now ready to fit our model to real data. I have chosen to look at the market "Khamenei out as Supreme Leader of Iran in 2025?"; daily prices are avaliable in `khamenei-out-as-supreme-leader-of-iran-in-2025-daily.csv`.

In [ ]:
import pandas as pd
data = pd.read_csv('khamenei-out-as-supreme-leader-of-iran-in-2025-daily.csv', names=['date', 'timestamp', 'yes_price'], skiprows=1)
data['no_price'] = 1 - data['yes_price']
data['date'] = pd.to_datetime(data['date'])
# normalize timestamp to [0, 1]. This is required by my implementation
data['timestamp'] -= data['timestamp'].iloc[0]
data['timestamp'] /= data['timestamp'].iloc[-1]
Delta = 0.002725 # the timedelta corresponding to daily prices
T = 1
T0 = 0
resolve_time = 1
n = len(data)
print(data.head())

# Create styled figure
fig, ax = create_styled_figure(figsize=(10, 5))

# Plot the data with golden yellow color
ax.plot(data['date'], data['yes_price']*100, color=COLORS['primary'], linewidth=2)

# Set title and labels
ax.set_title('Yes-Price for "Khamenei out as Supreme Leader of Iran in 2025?"', fontsize=FONT_SIZES['title'])
ax.set_ylabel('Price (US¢)', fontsize=FONT_SIZES['label'])
ax.set_ylim(-5, 105)

# Format x-axis with 3-letter month names
format_datetime_axis(ax)

# Adjust layout and save
plt.tight_layout()
save_styled_figure(fig, 'khameiniOut2025.svg')
plt.show()

We see the characteristic trend to zero, with a spike during the U.S.-Israeli attacks, as expected. Let's fit our model! First we do a grid search on $\kappa$ to find reasonable initial values:

In [ ]:
kappa_min, kappa_max = 1e-1, 5
grid_size = 10
theta0, best_val = initializer(grid_size, kappa_min, kappa_max, data['timestamp'], data['no_price'], Delta, T, resolve_time)

print(f'Result of Grid Search for κ ∈ [{kappa_min:.2f}, {kappa_max:.2f}]')
print('-' * 50)
print(f'Min. neg. log-likelihood: {best_val:.0f}')
print(f'μ: {theta0[0]:.2f}')
print(f'σ: {theta0[1]:.2f}')
print(f'κ: {theta0[2]:.2f}')

These initial values seem very reasonable. Let's use them to perform a full optimization:

In [ ]:
res = minimize(
    neg_ll,
    theta0,
    args=(data['timestamp'], data['no_price'], Delta, T, resolve_time),
    method="BFGS",
    options={"disp": False, "gtol": 1e-6}
)
mle_neg_ll = res.fun
mu_hat, log_sigma_hat, log_kappa_hat = res.x
sigma_hat = np.exp(log_sigma_hat)
kappa_hat = np.exp(log_kappa_hat)

# 95 % Wald CIs per the Delta method
mu_std, log_sigma_std, log_kappa_std = np.sqrt(np.diag(res.hess_inv))
z = 1.96
mu_95CI = [mu_hat - z*mu_std, mu_hat + z*mu_std]
log_sigma_95CI = [log(sigma_hat) - z*log_sigma_std, log(sigma_hat) + z*log_sigma_std]
log_kappa_95CI = [log(kappa_hat) - z*log_kappa_std, log(kappa_hat) + z*log_kappa_std]
sigma_95CI = np.exp(log_sigma_95CI)
kappa_95CI = np.exp(log_kappa_95CI)

jac_mu, jac_log_sigma, jac_log_kappa = res.jac

print("Parameter estimation summary (QMLE)")
print(f"Gradient norm at optimum: {np.linalg.norm(res.jac):.2e}\n")

print(f"{'Parameter':<10} {'Estimate':>12} {'95% Wald CI':>12} {'Jacobian':>12}")
print("-" * 62)

print(
    f"{'mu':<10} {mu_hat:12.2f} "
    f"[{mu_95CI[0]:.2f}, {mu_95CI[1]:.2f}] "
    f"{jac_mu:12.2e}"
)
print(
    f"{'sigma':<10} {sigma_hat:12.2f} "
    f"[{sigma_95CI[0]:.2f}, {sigma_95CI[1]:.2f}] "
    f"{jac_log_sigma:12.2e}"
)
print(
    f"{'kappa':<10} {kappa_hat:12.2f} "
    f"[{kappa_95CI[0]:.2f}, {kappa_95CI[1]:.2f}] "
    f"{jac_log_kappa:12.2e}\n"
)

Looks reasonable! Jacobians are basically zero given machine precision, so we should be at an optimum. The estimates are also robust to changes in initial values. Yay!

## Diagnostics
Let's first look at some goodnes-of-fit diagnostics:

In [ ]:
returns_norm = np.zeros(n-1)
for k in range(0, n-1):
    tk = Delta*k + T0
    tau = T - tk
    V = sigma_hat**2 / (2 * kappa_hat**3) * (2*kappa_hat*tau + 4*exp(-kappa_hat*tau) - exp(-2*kappa_hat*tau) - 3)
    L1 = kappa_hat / (exp(-kappa_hat*tau) - 1)
    L2 = mu_hat * tau - V / 2

    rk = data['no_price'][k+1] - data['no_price'][k]
    mean = data['no_price'][k] * Delta * (mu_hat + L1 * (log(data['no_price'][k]) + L2))
    std = abs(price[k] * sigma_hat / kappa_hat * sqrt(Delta) * np.expm1(-kappa_hat * tau))
    rk_norm = (rk - mean) / std

    returns_norm[k] = rk_norm

from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# histogram
axes[0, 0].hist(returns_norm, bins=30, density=True, alpha=0.6, color='g')
axes[0, 0].set_title('Standardized residuals', fontsize=22)
axes[0, 0].set_xlabel('Residual value', fontsize=18)
axes[0, 0].set_ylabel('Density', fontsize=18)
axes[0, 0].tick_params(axis="both", which="major", labelsize=16)

# QQ-plot
stats.probplot(returns_norm, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title("QQ-plot of Standardized Residuals", fontsize=22)
axes[0, 1].set_xlabel('Theoretical quantiles', fontsize=18)
axes[0, 1].set_ylabel('Ordered values', fontsize=18)
axes[0, 1].tick_params(axis="both", which="major", labelsize=16)

# ACF of residuals
nlags = min(40, len(returns_norm) - 1)
plot_acf(returns_norm, lags=nlags, ax=axes[1, 0])
axes[1, 0].set_title('ACF of standardized residuals', fontsize=22)
axes[1, 0].set_xlabel('Lags', fontsize=18)
axes[1, 0].set_ylabel('Autocorrelation', fontsize=18)
axes[1, 0].tick_params(axis="both", which="major", labelsize=16)

# ACF of squared residuals
plot_acf(returns_norm**2, lags=nlags, ax=axes[1, 1])
axes[1, 1].set_title('ACF of squared standardized residuals', fontsize=22)
axes[1, 1].set_xlabel('Lags', fontsize=18)
axes[1, 1].set_ylabel('Autocorrelation', fontsize=18)
axes[1, 1].tick_params(axis="both", which="major", labelsize=16)

plt.tight_layout()
path = Path.home() / 'Obsidian/Courses/MASM12 Nonlinear Timeseries/MASM12-project/diagnostics.pdf'
#fig.savefig(path)
plt.show()

# concise summary table
mean_all = np.mean(returns_norm)
median_all = np.median(returns_norm)
std_all = np.std(returns_norm)
n_total = returns_norm.size
pct_gt2 = 100 * np.mean(np.abs(returns_norm) > 2)

print("Standardized residuals summary")
print(f"{'Metric':<25}{'Value':>12}")
print("-" * 37)
print(f"{'Mean':<25}{mean_all:12.4f}")
print(f"{'Median':<25}{median_all:12.4f}")
print(f"{'Std':<25}{std_all:12.4f}")
print(f"{'Count':<25}{n_total:12d}")
print(f"{'>2σ (%)':<25}{pct_gt2:12.1f}")

fig, ax = plt.subplots(figsize=(12, 12))
plot_acf(returns_norm**2, lags=nlags, ax=ax)
ax.set_title('ACF of Squared Standardized Residuals', fontsize=22)
ax.set_xlabel('Lags', fontsize=18)
ax.set_ylabel('Autocorrelation', fontsize=18)
ax.set_ylim(-0.1, 1.1)
ax.tick_params(axis="both", which="major", labelsize=16)
plt.tight_layout()
plt.savefig('phd_presentation/acf_squared.pdf')
plt.show()

The mean and std look good! The QQ-plot is ok, with some deviation in the tails. The ACF of the straight residuals show a little bit of deviance but nothing major. Bit more problems with the ACF of the squared residuals, which show significant autocorrelation. This indicates that our model is not capturing all of the volatility dynamics in the data. Maybe we need a more complex volatility model? Not suprising with the big jump in the middle of the time series... We are not trying to model exogenous news shocks here.

<div class="alert alert-block alert-warning">
This is in-sample evaluation. Should do out-of-sample as well, but this is not that easy when all time-series are so short.
</div>

Let's also check the coverage of our one-step-ahead prediction intervals:

In [ ]:
def CI_coverage(level):
    alpha = 1 - level
    covered_count = 0
    tested_count = 0
    for k in range(0, n-1):
        tk = Delta*k + T0
        tau = T - tk
        V = sigma_hat**2 / (2 * kappa_hat**3) * (2*kappa_hat*tau + 4*exp(-kappa_hat*tau) - exp(-2*kappa_hat*tau) - 3)
        L1 = kappa_hat / (exp(-kappa_hat*tau) - 1)
        L2 = mu_hat * tau - V / 2

        if tk >= resolve_time: break

        mean = data['no_price'][k] + data['no_price'][k] * Delta * (mu_hat + L1 * (log(data['no_price'][k]) + L2))
        std = abs(data['no_price'][k] * sigma_hat / kappa_hat * sqrt(Delta) * np.expm1(-kappa_hat * tau))
        z = stats.norm.ppf(1 - alpha/2)

        tested_count += 1
        if data['no_price'][k+1] > mean - z*std and data['no_price'][k+1] < mean + z*std: covered_count += 1
    return covered_count / tested_count
    
levels = np.linspace(0.01, 0.99, 50)
coverages = [CI_coverage(level) for level in levels]

plt.figure(figsize=(8, 8))
plt.plot(levels, coverages, label='Empirical coverage', linewidth=4)
plt.plot(levels, levels, 'k--', label='Perfect calibration', linewidth=4)
plt.xlabel('Nominal coverage level', fontsize=18)
plt.ylabel('Empirical coverage', fontsize=18)
plt.title('Prediction Interval Coverage', fontsize=22)
plt.legend(fontsize=18)
plt.grid(True, alpha=0.3)
plt.tick_params(axis="both", which="major", labelsize=16)
path = Path.home() / 'Obsidian/Courses/MASM12 Nonlinear Timeseries/MASM12-project/ci_coverage.pdf'
#plt.savefig(path)
plt.show()

We systematically oversetimate volatility, leading to overcoverage.

## Reconstructing Intensity
With parameter estimates we can now reconstruct the intensity from the price.

In [ ]:
intensity_hat = np.zeros(n)
intensity_hat[-1] = np.nan
for k, (tk, price_k) in enumerate(zip(data['timestamp'][:-1], data['no_price'][:-1])):
    tau = T - tk
    V = sigma_hat**2 / (2 * kappa_hat**3) * (2*kappa_hat*tau + 4*exp(-kappa_hat*tau) -exp(-2*kappa_hat*tau) - 3)
    L1 = kappa_hat / np.expm1(-kappa_hat*tau)
    L2 = mu_hat * tau - V / 2
    intensity_hat[k] = mu_hat + L1 * (log(price_k) + L2)
data['intensity'] = intensity_hat

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(10, 6),
    gridspec_kw={"height_ratios": [1, 1]}
)
fig.suptitle(f'''Intensity Reconstruction for "Khamenei out as Supreme Leader of Iran in 2025?"
μ̂={mu:.2f}, σ̂={sigma:.2f}, κ̂={kappa:.2f}, n={n}''')

ax1.plot(data['date'], data['yes_price'] * 100)
ax1.set_title('Yes-Contract Price')
ax1.set_ylabel('Price  (US¢)')
ax1.set_ylim(-5, 105)

ax2.plot(data['date'], data['intensity'])
ax2.set_title('Reconstructed Intensity')
ax2.set_ylabel('Intensity')

plt.tight_layout()
plt.show()

## Reconstructing Volatility
Next step is to reconstruct the instantanous volatility of the prices.

In [ ]:
vol_hat = np.zeros(n)
for k, (tk, price_k) in enumerate(zip(data['timestamp'], data['no_price'])):
    tau = T - tk
    vol_hat[k] = abs(price_k * sigma_hat / kappa_hat * np.expm1(-kappa_hat*tau))
vol_realized = np.sqrt(np.pi / 2) * abs(np.diff(data['no_price'])) / sqrt(Delta)
data['vol_reconstructed'] = vol_hat
data['vol_realized'] = np.append(vol_realized, np.nan)

fig, (ax0, ax1, ax2) = plt.subplots(
    3, 1,
    sharex=True,
    figsize=(10, 9),
    gridspec_kw={"height_ratios": [1, 1, 1]}
)
fig.suptitle(f'Reconstructed Intensity and Volatility for\n"Khamenei out as Supreme Leader of Iran in 2025?"', fontsize=22)
# --- Top: yes price ---
ax0.plot(data['date'], data['yes_price'] * 100, linewidth=4)
ax0.set_title('Yes-Contract Price', fontsize=18)
ax0.set_ylabel('Price (US¢)', fontsize=18)
ax0.set_ylim(-5, 105)
ax0.tick_params(axis='both', which='major', labelsize=16)

# --- Middle: intensity ---
ax1.plot(data['date'], data['intensity'], linewidth=4)
ax1.set_title('Intensity Reconstruction', fontsize=18)
ax1.set_ylabel('Intensity', fontsize=18)
ax1.tick_params(axis='both', which='major', labelsize=16)

# --- Bottom: volatility ---
ax2.plot(data['date'], data['vol_realized'], color='green', label='Realized', alpha=0.3, linewidth=4)
ax2.plot(data['date'], data['vol_reconstructed'], label='Reconstructed', linewidth=4)
ax2.set_title('Volatility Reconstruction', fontsize=18)
ax2.set_ylabel(r'Volatility', fontsize=18)
ax2.legend(fontsize=18)
ax2.tick_params(axis='both', which='major', labelsize=16)
from pathlib import Path

path = Path.home() / 'Obsidian/Courses/MASM12 Nonlinear Timeseries/MASM12-project/reconstruction.pdf'
#fig.savefig(path)
fig.tight_layout()
plt.show()

What do we make of the fact that the RV spikes when the reconstructed volatility dips? Not sure...

## Probability of Resolution Within a Given Horizon
First try on the parameter estimates fitted on the entire series:

In [ ]:
i0 = 250
t0 = data['timestamp'].loc[i0]
price0 = data['yes_price'].loc[i0]
num_mc_sims = 1000

res_times = np.zeros(num_mc_sims)
for k in range(num_mc_sims):
    _, _, _, res_t, _ = simulate_timing_market(mu_hat, sigma_hat, kappa_hat, price0, n-i0, T0=t0, T=T)
    if res_t is None:
        res_times[k] = np.inf
    else:
        res_times[k] = res_t
        
horizon_grid = np.linspace(0, T-t0, num=1000)
prob_grid = [100*np.mean(res_times <= t0 + horizon) for horizon in horizon_grid]

plt.plot(horizon_grid, prob_grid)
plt.axhline(price0*100, color="grey", linestyle='--')
plt.ylabel('Cumulative Probability of Resolution (%)')
#plt.ylim(-5, 105)
plt.xlabel(f'Horizon (time)')
plt.title(rf'''Model-Implied Probability of Market Resolution of
"Khamenei out as Supreme Leader of Iran in 2025?"
Within Horizon From $t_0 = {t0:.2f}$
(Market Price at time $t_0$ in Grey)''')
plt.show()

This is not very well calibrated! Strange...

To do this properly, we should refit the model with data only avaliable up to, say, the first 250 days.

In [ ]:
kappa_min, kappa_max = 1e-1, 5
grid_size = 10
theta0, best_val = initializer(grid_size, kappa_min, kappa_max, data['timestamp'].iloc[:i0], data['no_price'].iloc[:i0], Delta, T, resolve_time)

print(f'Result of Grid Search for κ ∈ [{kappa_min:.2f}, {kappa_max:.2f}]')
print('-' * 50)
print(f'Min. neg. log-likelihood: {best_val:.0f}')
print(f'μ: {theta0[0]:.2f}')
print(f'σ: {theta0[1]:.2f}')
print(f'κ: {theta0[2]:.2f}')

res = minimize(
    neg_ll,
    theta0,
    args=(data['timestamp'].iloc[:i0], data['no_price'].iloc[:i0], Delta, T, resolve_time),
    method="BFGS",
    options={"disp": False, "gtol": 1e-6}
)
mle_neg_ll = res.fun
mu_hat, log_sigma_hat, log_kappa_hat = res.x
sigma_hat = np.exp(log_sigma_hat)
kappa_hat = np.exp(log_kappa_hat)

# 95 % Wald CIs per the Delta method
mu_std, log_sigma_std, log_kappa_std = np.sqrt(np.diag(res.hess_inv))
z = 1.96
mu_95CI = [mu_hat - z*mu_std, mu_hat + z*mu_std]
log_sigma_95CI = [log(sigma_hat) - z*log_sigma_std, log(sigma_hat) + z*log_sigma_std]
log_kappa_95CI = [log(kappa_hat) - z*log_kappa_std, log(kappa_hat) + z*log_kappa_std]
sigma_95CI = np.exp(log_sigma_95CI)
kappa_95CI = np.exp(log_kappa_95CI)

jac_mu, jac_log_sigma, jac_log_kappa = res.jac

print("Parameter estimation summary (QMLE)")
print(f"Gradient norm at optimum: {np.linalg.norm(res.jac):.2e}\n")

print(f"{'Parameter':<10} {'Estimate':>12} {'95% Wald CI':>12} {'Jacobian':>12}")
print("-" * 62)

print(
    f"{'mu':<10} {mu_hat:12.2f} "
    f"[{mu_95CI[0]:.2f}, {mu_95CI[1]:.2f}] "
    f"{jac_mu:12.2e}"
)
print(
    f"{'sigma':<10} {sigma_hat:12.2f} "
    f"[{sigma_95CI[0]:.2f}, {sigma_95CI[1]:.2f}] "
    f"{jac_log_sigma:12.2e}"
)
print(
    f"{'kappa':<10} {kappa_hat:12.2f} "
    f"[{kappa_95CI[0]:.2f}, {kappa_95CI[1]:.2f}] "
    f"{jac_log_kappa:12.2e}\n"
)

These estimates are very different from those with the full data! The Jacobians look good, but both $\kappa$ and $\sigma$ are very high... Let's try the probability estimates anyway.

In [ ]:
t0 = data['timestamp'].loc[i0]
price0 = data['yes_price'].loc[i0]
num_mc_sims = 1000

res_times = np.zeros(num_mc_sims)
for k in range(num_mc_sims):
    _, _, _, res_t, _ = simulate_timing_market(mu_hat, sigma_hat, kappa_hat, price0, n-i0, T0=t0, T=T)
    if res_t is None:
        res_times[k] = np.inf
    else:
        res_times[k] = res_t
        
horizon_grid = np.linspace(0, T-t0, num=1000)
prob_grid = [100*np.mean(res_times <= t0 + horizon) for horizon in horizon_grid]

plt.figure(figsize=(9, 9))
plt.plot(horizon_grid/Delta, prob_grid, linewidth=4)
plt.axhline(price0*100, color="grey", linestyle='--', label='Market Price at $t_0$', linewidth=4)
plt.ylabel('Cumulative Probability of Resolution (%)', fontsize=18)
plt.xlabel(f'Horizon (days)', fontsize=18)
plt.title(rf'''Probability of Market Resolution for
"Khamenei out in 2025?"
Within Horizon From $t_0 = {data['date'][i0].strftime("%b. %d")}$''', fontsize=22)
plt.legend(fontsize=18)
plt.tick_params(axis="both", which="major", labelsize=16)
path = Path.home() / 'Obsidian/Courses/MASM12 Nonlinear Timeseries/MASM12-project/prob_real.pdf'
#plt.savefig(path)
plt.show()

Horrible calibration! I wonder what makes the model self-inconsistent like this... I don't think its poor parameter estimation, because this exact setup woked fine on simulated data. Maybe its a goodess-of-fit issue...

A lot more simulated paths resolve before $T$ than is implied by the market price at $t_0$. This implies an overestimation of the latent intensity. Could it be that the OU-paramters are overblown to fit jumps and clustered volatility. This could probably be accounted for by allowing either jumps or time-varying paramters. Future work!

## Exact Fine-Grained Prediction Probabilities

In [ ]:
t0 = data['timestamp'].loc[i0]
price0 = data['yes_price'].loc[i0]
num_mc_sims = 1000

res_times = np.zeros(num_mc_sims)
for k in range(num_mc_sims):
    _, _, _, res_t, _ = simulate_timing_market(mu_hat, sigma_hat, kappa_hat, price0, n-i0, T0=t0, T=T)
    if res_t is None:
        res_times[k] = np.inf
    else:
        res_times[k] = res_t
        
horizon_grid = np.linspace(0, T-t0, num=1000)
prob_grid_mc = [100*np.mean(res_times <= t0 + horizon) for horizon in horizon_grid]

exact_prob_grid = [100*
    compute_exact_resolution_probability(
        1-price0, horizon, mu_hat, kappa_hat, sigma_hat, T, t0
    ) for horizon in horizon_grid
]


plt.figure(figsize=(10, 6))
plt.plot(horizon_grid, exact_prob_grid, 'b-', linewidth=2, label='Exact Solution')
plt.plot(horizon_grid, prob_grid_mc, 'r--', linewidth=2, label='Monte Carlo')
plt.axhline(price0*100, color="grey", linestyle='--', label='Market Price at $t_0$', linewidth=1)
plt.xlabel('Time Horizon')
plt.ylabel('Probability of Resolution')
plt.title(f'Comparison of Exact vs Monte Carlo Resolution Probabilities\n(Current time t₀ = {t0:.2f}, Current price = {price0:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Forecasting Future Prices, Intensities, and Volatilities
Finally, we can use the MC setup to forecast future prices, intensities, and volatilities. We again use paramteres estimated only on the data avaliable at the prediction time.

In [ ]:
num_mc_sims = 1000
horizon = 0.1
horizon_index = i0 + int(horizon/Delta) + 1

t_MC = np.zeros([num_mc_sims, n-i0])
price_MC = np.zeros([num_mc_sims, n-i0])
intensity_MC = np.zeros([num_mc_sims, n-i0])
vol_MC = np.zeros([num_mc_sims, n-i0])
resolve_time_MC = np.zeros(num_mc_sims)
Delta_MC = np.zeros(num_mc_sims)
for i in range(num_mc_sims):
    while True:
        t_sample, price_sample, intensity_sample, resolve_time_sample, Delta_sample = simulate_timing_market(mu_hat, sigma_hat, kappa_hat, price0, n-i0, T0=t0, T=T)
        if resolve_time_sample is None: resolve_time_sample = np.inf
        if resolve_time_sample >= t0+horizon: break
        
    for k, (tk, price_k) in enumerate(zip(t_sample, price_sample)):
        tau = T - tk
        vol_MC[i, k] = abs(price_k * sigma_hat / kappa_hat * np.expm1(-kappa_hat*tau))
    
    t_MC[i, :] = t_sample
    price_MC[i, :] = price_sample
    intensity_MC[i, :] = intensity_sample
    resolve_time_MC[i] = resolve_time_sample
    Delta_MC[i] = Delta_sample

price_mean = np.mean(price_MC, axis=0)
price_low, price_high = np.quantile(price_MC, [0.025, 0.975], axis=0)
intensity_mean = np.mean(intensity_MC, axis=0)
intensity_low, intensity_high = np.quantile(intensity_MC, [0.025, 0.975], axis=0)
vol_mean = np.mean(vol_MC, axis=0)
vol_low, vol_high = np.quantile(vol_MC, [0.025, 0.975], axis=0)

# reconstruct volatility with parameter estimated on partial data
vol_hat = np.zeros(n)
for k, (tk, price_k) in enumerate(zip(data['timestamp'], data['no_price'])):
    tau = T - tk
    vol_hat[k] = abs(price_k * sigma_hat / kappa_hat * np.expm1(-kappa_hat*tau))

# reconsturct intensity with parameter estimated on partial data
intensity_hat = np.zeros(n)
intensity_hat[-1] = np.nan
for k, (tk, price_k) in enumerate(zip(data['timestamp'][:-1], data['no_price'][:-1])):
    tau = T - tk
    V = sigma_hat**2 / (2 * kappa_hat**3) * (2*kappa_hat*tau + 4*exp(-kappa_hat*tau) -exp(-2*kappa_hat*tau) - 3)
    L1 = kappa_hat / np.expm1(-kappa_hat*tau)
    L2 = mu_hat * tau - V / 2
    intensity_hat[k] = mu_hat + L1 * (log(price_k) + L2)

fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1,
    sharex=True,
    figsize=(12, 8),
    gridspec_kw={"height_ratios": [1, 1, 1]}
)
fig.suptitle(f'''Conditional Monte Carlo Forecasts for 
"Khamenei out as Supreme Leader of Iran in 2025?"''',fontsize=22)

ax1.axvline(data['date'][i0], color='grey', linestyle='--')
ax1.plot(data['date'][:horizon_index], (data['yes_price'][:horizon_index]) * 100, label='True')
ax1.plot(data['date'][i0:horizon_index], (1-price_mean[:horizon_index-i0])*100, color='orange', label='MC Prediction')
ax1.fill_between(data['date'][i0:horizon_index], 100*(1-price_low[:horizon_index-i0]), 100*(1-price_high[:horizon_index-i0]), color='orange', alpha=0.5)
ax1.set_title('Contract Price With MC Predictions', fontsize=18)
ax1.set_ylabel('Price  (US¢)', fontsize=18)
ax1.set_ylim(-5, 105)
ax1.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=18
)

ax2.axvline(data['date'][i0], color='grey', linestyle='--')
ax2.plot(data['date'][:horizon_index], data['intensity'][:horizon_index], label='Reconstructed (full)')
ax2.plot(data['date'][:horizon_index], intensity_hat[:horizon_index], color='green', label='Reconstructed (partial)')
ax2.plot(
    data['date'][i0:horizon_index],
    intensity_mean[:horizon_index-i0],
    color='orange',
    label='MC Prediction'
)
ax2.fill_between(
    data['date'][i0:horizon_index],
    intensity_low[:horizon_index-i0],
    intensity_high[:horizon_index-i0],
    color='orange',
    alpha=0.5
)
ax2.set_title('Intensity With MC Predictions', fontsize=18)
ax2.set_ylabel('Intensity', fontsize=18)
ax2.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=18
)

ax3.axvline(data['date'][i0], color='grey', linestyle='--')
ax3.plot(data['date'][:horizon_index], data['vol_reconstructed'][:horizon_index], label='Reconstructed (full)')
ax3.plot(data['date'][:horizon_index], vol_hat[:horizon_index], color='green', label='Reconstructed (partial)')
ax3.plot(
    data['date'][i0:horizon_index],
    vol_mean[:horizon_index-i0],
    color='orange',
    label='MC Prediction'
)
ax3.fill_between(
    data['date'][i0:horizon_index],
    vol_low[:horizon_index-i0],
    vol_high[:horizon_index-i0],
    color='orange',
    alpha=0.5
)
ax3.set_title('Volatility With MC Predictions', fontsize=18)
ax3.set_ylabel('Volatility', fontsize=18)
ax3.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=18
)

for ax in (ax1, ax2, ax3):
    ax.tick_params(axis='both', labelsize=16)

plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.tight_layout()
path = Path.home() / 'Obsidian/Courses/MASM12 Nonlinear Timeseries/MASM12-project/forecasts.pdf'
#fig.savefig(path)
plt.show()

horizon/Delta

This is suprisingly good, espeicially the price forecast! Intensity is very uncertain. The dynamics of the volatility prediction seems ok, but the starting point is of compared to the reconstruction using the full data. Not very strange; just a consequence of the two parameter estimates being very different. Overall quite happy with this!
